# Powerlifting — Pipeline GPU v2 (Google Colab)

## Historique
| Version | CV RMSE | Public LB |
|---|---|---|
| sklearn GradientBoosting | 82.60 | ~80 |
| Blend LGBM + CatBoost (local) | 77.77 | — |
| v1 GPU : blend XGB + CatBoost + MLP (5-fold, 1 seed) | 77.775 | 78.0 |

## Changements v2
| # | Changement | Pourquoi |
|---|---|---|
| 1 | **+ LightGBM** dans le blend (4 modèles) | meilleur modèle seul en local (78.36), absent de la v1 |
| 2 | **10-fold × 3 seeds** | +10 % de données par modèle, moyenne de 30 modèles → moins de variance |
| 3 | **Optuna** (optionnel, `TUNE=True`) pour XGBoost et CatBoost | réglage des hyperparamètres sur GPU |
| 4 | **MLP renforcé** : rank-gauss sur les numériques, réseau plus large, plus d'epochs | meilleure diversité pour le blend |
| 5 | **Sauvegarde par fold** sur le Drive | une déconnexion Colab ne fait perdre que le fold en cours |

Pistes écartées (testées) : FE supplémentaire, target encoding, pseudo-identifiant d'athlète, données externes (interdites).

## Utilisation
1. Le dossier est dans `MyDrive/Colab Notebooks/performances-en-powerlifting`.
2. `Exécution → Modifier le type d'exécution → GPU (T4)` puis `Tout exécuter`.
3. Si la session coupe : relancer `Tout exécuter`, les folds déjà calculés sont rechargés.
4. Soumettre `outputs_v2/submission_FINAL_blend.csv`.

## 0. Configuration

In [ ]:
!pip install -q catboost optuna
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, time, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import KFold

# ---------------- Paramètres ----------------
DRIVE_FOLDER = 'Colab Notebooks/performances-en-powerlifting'   # dossier dans MyDrive
N_FOLDS = 10
SEEDS = [42, 7, 2024]
# LightGBM tourne sur CPU (Colab = 2 cœurs) → 1 seed par défaut pour limiter le temps
MODEL_SEEDS = dict(lgb=[42], xgb=SEEDS, catboost=SEEDS, mlp=SEEDS)
RUN = dict(lgb=True, xgb=True, catboost=True, mlp=True)
TUNE = False               # True = recherche Optuna pour XGBoost et CatBoost avant l'entraînement
TUNE_TRIALS = 40           # essais max par modèle
TUNE_TIMEOUT = 3600        # secondes max par modèle
FORCE_RETRAIN = False      # True = ignorer les folds sauvegardés
DEBUG = False              # True = petit échantillon (test rapide du code)
# --------------------------------------------

if DEBUG:
    N_FOLDS, SEEDS, TUNE_TRIALS = 3, [42], 2
    MODEL_SEEDS = {k: [42] for k in MODEL_SEEDS}

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
except ImportError:
    DATA_DIR = '.'

assert os.path.exists(os.path.join(DATA_DIR, 'train.csv')), f'train.csv introuvable dans {DATA_DIR}'
OUT_DIR = os.path.join(DATA_DIR, 'outputs_v2_debug' if DEBUG else 'outputs_v2')
os.makedirs(OUT_DIR, exist_ok=True)

try:
    import torch
    GPU = torch.cuda.is_available()
except ImportError:
    GPU = False
DEVICE = 'cuda' if GPU else 'cpu'

N_EST = 100 if DEBUG else 20000   # arbres max (early stopping)
EPOCHS = 2 if DEBUG else 40       # MLP

print('DATA_DIR :', DATA_DIR)
print('OUT_DIR  :', OUT_DIR)
print('GPU      :', GPU)
print(f'{N_FOLDS}-fold, seeds par modèle : {MODEL_SEEDS}')

## 1. Lecture des données

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
if DEBUG:
    train = train.sample(3000, random_state=0).reset_index(drop=True)
    test = test.head(1000).reset_index(drop=True)

TARGET = 'TotalKg'
y = train[TARGET].values.astype(np.float64)
print('Train:', train.shape, '| Test:', test.shape)

## 2. Feature engineering
Seul le groupe *weight class* a amélioré la CV lors de l'ablation (−0.19). Les catégorielles sont gérées nativement par les modèles.

In [ ]:
RAW_NUM = ['Age', 'BodyweightKg', 'MeetYear']
CAT = [c for c in test.columns if test[c].dtype == 'object']

def add_features(df):
    out = df.copy()
    out['WeightClass_num'] = pd.to_numeric(out['WeightClassKg'].str.replace('+', '', regex=False), errors='coerce')
    out['Is_SHW'] = out['WeightClassKg'].str.endswith('+').astype(int)
    out['BW_minus_Class'] = out['BodyweightKg'] - out['WeightClass_num']
    return out

NUM = RAW_NUM + ['WeightClass_num', 'Is_SHW', 'BW_minus_Class']
FEATS = NUM + CAT

full = pd.concat([add_features(train)[FEATS], add_features(test)[FEATS]], ignore_index=True)
for c in CAT:
    full[c] = full[c].astype('category')
X  = full.iloc[:len(train)].reset_index(drop=True)
Xt = full.iloc[len(train):].reset_index(drop=True)
print(f'{len(NUM)} numériques + {len(CAT)} catégorielles = {len(FEATS)} features')

## Outils : CV avec sauvegarde par fold, chargement des paramètres Optuna

In [ ]:
results = {}  # nom -> (oof, pred_test, rmse)

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def save_submission(fname, pred):
    path = os.path.join(OUT_DIR, fname)
    pd.DataFrame({'id': test['id'], TARGET: np.clip(pred, 0, None)}).to_csv(path, index=False)
    print('Soumission :', path)

def run_cv(name, fit_predict, seeds):
    """fit_predict(idx_train, idx_val, seed) -> (pred_val, pred_test). Chaque fold est sauvegardé."""
    oof, pred = np.zeros(len(X)), np.zeros(len(Xt))
    t0 = time.time()
    for seed in seeds:
        for i, (a, b) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(X)):
            f = os.path.join(OUT_DIR, f'{name}_s{seed}_f{i}.npz')
            if os.path.exists(f) and not FORCE_RETRAIN:
                d = np.load(f)
                p_val, p_test, tag = d['val'], d['test'], 'cache'
            else:
                t = time.time()
                p_val, p_test = fit_predict(a, b, seed)
                np.savez(f, val=p_val, test=p_test)
                tag = f'{time.time() - t:.0f}s'
            oof[b] += p_val / len(seeds)
            pred += p_test / (N_FOLDS * len(seeds))
            print(f'  [{name}] seed {seed} fold {i}: RMSE {rmse(y[b], p_val):.3f}  ({tag})')
    results[name] = (oof, pred, rmse(y, oof))
    print(f'[{name}] CV RMSE = {results[name][2]:.3f}   ({(time.time() - t0) / 60:.1f} min)')
    save_submission(f'submission_{name}.csv', pred)

# Split unique (80/20) utilisé par Optuna
TUNE_A, TUNE_B = next(KFold(5, shuffle=True, random_state=123).split(X))

def tune(name, objective, default_params):
    """Optuna si TUNE et pas déjà fait ; renvoie (params, nom_du_modèle)."""
    f = os.path.join(OUT_DIR, f'best_params_{name}.json')
    if TUNE and not os.path.exists(f):
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=0))
        study.enqueue_trial(default_params)   # 1er essai = paramètres par défaut
        study.optimize(objective, n_trials=TUNE_TRIALS, timeout=TUNE_TIMEOUT,
                       callbacks=[lambda s, t: print(f'  trial {t.number}: {t.value:.3f}  (best {s.best_value:.3f})')])
        json.dump(study.best_params, open(f, 'w'), indent=1)
    if os.path.exists(f):
        params = {**default_params, **json.load(open(f))}
        print(f'[{name}] paramètres Optuna :', params)
        return params, f'{name}_tuned'
    return default_params, name

## 3. LightGBM (CPU)
Mêmes paramètres que la meilleure version locale (CV 78.36).

In [ ]:
import lightgbm as lgb

LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, cat_smooth=20)

def lgb_fit(a, b, seed):
    m = lgb.LGBMRegressor(n_estimators=N_EST, random_state=seed, n_jobs=-1, verbose=-1, **LGB_PARAMS)
    m.fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], callbacks=[lgb.early_stopping(200, verbose=False)])
    return m.predict(X.iloc[b]), m.predict(Xt)

if RUN['lgb']:
    run_cv('lgb', lgb_fit, MODEL_SEEDS['lgb'])

## 4. XGBoost (GPU) + Optuna optionnel

In [ ]:
import xgboost as xgb
print('xgboost', xgb.__version__)

XGB_DEFAULT = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8,
                   colsample_bytree=0.7, reg_lambda=1.0, reg_alpha=0.001, gamma=0.0)

def xgb_train(a, b, seed, params, with_test=True):
    m = xgb.XGBRegressor(n_estimators=N_EST, tree_method='hist', device=DEVICE, enable_categorical=True,
                         max_cat_to_onehot=1, early_stopping_rounds=200, eval_metric='rmse',
                         random_state=seed, **params)
    m.fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], verbose=False)
    return m.predict(X.iloc[b]), (m.predict(Xt) if with_test else None)

def xgb_objective(trial):
    p = dict(
        learning_rate=trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        max_depth=trial.suggest_int('max_depth', 4, 10),
        min_child_weight=trial.suggest_float('min_child_weight', 1, 100, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.4, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 20, log=True),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 20, log=True),
        gamma=trial.suggest_float('gamma', 0, 5),
    )
    p_val, _ = xgb_train(TUNE_A, TUNE_B, 0, p, with_test=False)
    return rmse(y[TUNE_B], p_val)

XGB_PARAMS, XGB_NAME = tune('xgb', xgb_objective, XGB_DEFAULT)

if RUN['xgb']:
    run_cv(XGB_NAME, lambda a, b, s: xgb_train(a, b, s, XGB_PARAMS), MODEL_SEEDS['xgb'])

## 5. CatBoost (GPU) + Optuna optionnel

In [ ]:
from catboost import CatBoostRegressor

X_cb  = X.astype({c: str for c in CAT})
Xt_cb = Xt.astype({c: str for c in CAT})

CB_DEFAULT = dict(learning_rate=0.05, depth=8, l2_leaf_reg=3.0, random_strength=1.0,
                  bagging_temperature=1.0, border_count=128)

def cb_train(a, b, seed, params, with_test=True):
    extra = dict(task_type='GPU', devices='0') if GPU else {}
    m = CatBoostRegressor(iterations=N_EST, loss_function='RMSE', od_type='Iter', od_wait=300,
                          random_seed=seed, verbose=0, **extra, **params)
    m.fit(X_cb.iloc[a], y[a], cat_features=CAT, eval_set=(X_cb.iloc[b], y[b]), use_best_model=True)
    return m.predict(X_cb.iloc[b]), (m.predict(Xt_cb) if with_test else None)

def cb_objective(trial):
    p = dict(
        learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        depth=trial.suggest_int('depth', 4, 9),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 30, log=True),
        random_strength=trial.suggest_float('random_strength', 0, 3),
        bagging_temperature=trial.suggest_float('bagging_temperature', 0, 2),
        border_count=trial.suggest_categorical('border_count', [128, 254]),
    )
    p_val, _ = cb_train(TUNE_A, TUNE_B, 0, p, with_test=False)
    return rmse(y[TUNE_B], p_val)

CB_PARAMS, CB_NAME = tune('catboost', cb_objective, CB_DEFAULT)

if RUN['catboost']:
    run_cv(CB_NAME, lambda a, b, s: cb_train(a, b, s, CB_PARAMS), MODEL_SEEDS['catboost'])

## 6. MLP PyTorch (GPU) — version renforcée
- Numériques : standardisation **+ rank-gauss** (QuantileTransformer → loi normale) + indicateurs de NaN
- Embeddings de taille `min(48, 1.6·n^0.56)` ; modalités rares (< 5 dans le train) regroupées
- Réseau 1024 → 512 → 256, BatchNorm + SiLU + Dropout, AdamW + OneCycle, 40 epochs
- Meilleure epoch retenue sur le fold de validation

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import QuantileTransformer

n_tr = len(X)
cat_codes, cards = [], []
for c in CAT:
    codes = full[c].cat.codes.values.astype(np.int64) + 1          # NaN -> 0
    counts = np.bincount(codes[:n_tr], minlength=codes.max() + 1)
    codes = np.where(counts[codes] >= 5, codes, 0)                  # rares -> 0
    _, codes = np.unique(codes, return_inverse=True)
    cat_codes.append(codes); cards.append(int(codes.max()) + 1)
XC = np.stack(cat_codes, axis=1)

num = full[NUM].astype(float)
nan_ind = num[[c for c in NUM if num[c].isna().any()]].isna().astype(float).add_suffix('_nan')
num = num.fillna(num.median())
num_std = (num - num.mean()) / (num.std() + 1e-9)
num_rg = pd.DataFrame(QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=0)
                      .fit_transform(num), columns=[f'{c}_rg' for c in NUM])
XN = pd.concat([num_std, num_rg, nan_ind], axis=1).values.astype(np.float32)

XC_t = torch.tensor(XC, device=DEVICE)
XN_t = torch.tensor(XN, device=DEVICE)
print('Cardinalités :', dict(zip(CAT, cards)))
print('Entrées numériques NN :', XN.shape[1])


class TabMLP(nn.Module):
    def __init__(self, cards, n_num):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, max(2, min(48, round(1.6 * n ** 0.56)))) for n in cards])
        d = sum(e.embedding_dim for e in self.embs) + n_num
        block = lambda i, o, p: [nn.Linear(i, o), nn.BatchNorm1d(o), nn.SiLU(), nn.Dropout(p)]
        self.mlp = nn.Sequential(*block(d, 1024, 0.2), *block(1024, 512, 0.15), *block(512, 256, 0.1),
                                 nn.Linear(256, 1))

    def forward(self, xc, xn):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.mlp(torch.cat(e + [xn], dim=1)).squeeze(1)


@torch.no_grad()
def nn_predict(model, idx, bs=8192):
    model.eval()
    return torch.cat([model(XC_t[idx[i:i + bs]], XN_t[idx[i:i + bs]]) for i in range(0, len(idx), bs)]).cpu().numpy()


def mlp_fit(a, b, seed, bs=512, lr=2e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    mu, sd = y[a].mean(), y[a].std()
    y_t = torch.tensor((y - mu) / sd, dtype=torch.float32, device=DEVICE)
    a_t, b_t = torch.tensor(a, device=DEVICE), torch.tensor(b, device=DEVICE)
    test_idx = torch.arange(n_tr, len(full), device=DEVICE)

    model = TabMLP(cards, XN.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps_per_epoch = (len(a) + bs - 1) // bs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * steps_per_epoch)
    loss_fn = nn.MSELoss()

    best, best_state = np.inf, None
    for ep in range(EPOCHS):
        model.train()
        perm = a_t[torch.randperm(len(a_t), device=DEVICE)]
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            if len(idx) < 2:   # BatchNorm
                sched.step(); continue
            opt.zero_grad()
            loss_fn(model(XC_t[idx], XN_t[idx]), y_t[idx]).backward()
            opt.step(); sched.step()
        val = rmse(y[b], nn_predict(model, b_t) * sd + mu)
        if val < best:
            best, best_state = val, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return nn_predict(model, b_t) * sd + mu, nn_predict(model, test_idx) * sd + mu

if RUN['mlp']:
    run_cv('mlp', mlp_fit, MODEL_SEEDS['mlp'])

## 7. Blend final
Poids ≥ 0 par moindres carrés non négatifs (NNLS) sur les OOF.

In [ ]:
from scipy.optimize import nnls

names = list(results)
P_oof  = np.column_stack([results[n][0] for n in names])
P_test = np.column_stack([results[n][1] for n in names])

w, _ = nnls(P_oof, y)
oof_blend, pred_blend = P_oof @ w, P_test @ w
results['blend'] = (oof_blend, pred_blend, rmse(y, oof_blend))

print('Poids :', dict(zip(names, np.round(w, 3))))
print(f'Blend CV RMSE = {results["blend"][2]:.3f}')
save_submission('submission_FINAL_blend.csv', pred_blend)

In [ ]:
print('Corrélation des OOF :')
display(pd.DataFrame(P_oof, columns=names).corr().round(4))

summary = pd.Series({'v1 blend (ref, LB 78.0)': 77.775, **{k: v[2] for k, v in results.items()}}).sort_values()
print(summary.round(3))

fig, ax = plt.subplots(figsize=(8, 0.5 * len(summary) + 1))
ax.barh(summary.index, summary.values,
        color=['#70AD47' if k == 'blend' else '#BFBFBF' if 'ref' in k else '#5B9BD5' for k in summary.index])
for i, v in enumerate(summary.values):
    ax.text(v + 0.05, i, f'{v:.3f}', va='center')
ax.set_xlim(summary.min() - 2, summary.max() + 1)
ax.set_xlabel('CV RMSE (kg)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()